# Notebook 6 - Large LLM Comparison on Google Colab

## Goal

This notebook is designed for Google Colab Pro or Pro+. It compares the same truth-probing experiment across multiple LLMs.

The professor asked for different results on different LLMs. This notebook produces the table needed for that comparison.

## What this notebook compares

Default enabled models:

- `microsoft/phi-2` as the local baseline,
- `mistralai/Mistral-7B-Instruct-v0.3`,
- `meta-llama/Meta-Llama-3-8B-Instruct`.

Optional 70B models:

- `meta-llama/Llama-3.3-70B-Instruct`,
- `meta-llama/Meta-Llama-3-70B-Instruct`.

The 70B models are not enabled automatically unless the runtime has enough GPU memory. They require a Hugging Face token and accepted Meta model access.


## Step 1 - Runtime setup

In Colab, select a GPU runtime before running this notebook:

`Runtime -> Change runtime type -> Hardware accelerator -> GPU`

For 7B/8B models, a T4/L4/A100 runtime with 4-bit loading is usually enough. For 70B models, use the largest available high-memory GPU. If the 70B model does not fit, the notebook catches the error and continues with the other models.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/imadsharof/Lie-Detector-for-LLM.git"
REPO_DIR = Path("/content/Lie-Detector-for-LLM")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".[colab]",
        ],
        check=True,
    )
else:
    print("Not running inside Colab. Using the current local repository.")

## Step 2 - Import the package and inspect the GPU

The experiment needs access to hidden states, so it loads models through Hugging Face Transformers rather than using an API endpoint.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root. Run this notebook from the repository.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU for large-model comparison.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU name        : {GPU_NAME}")
print(f"GPU memory (GB) : {GPU_TOTAL_GB:.1f}")


## Step 3 - Hugging Face login

Llama models are gated. Before running this notebook:

1. create a Hugging Face account,
2. accept the license for the Llama model pages you want to use,
3. create an access token,
4. store it in Colab secrets as `HF_TOKEN` or paste it when prompted.


In [ ]:
import os
from huggingface_hub import login, notebook_login

hf_token = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face with HF_TOKEN.")
else:
    print("No HF_TOKEN found. A login widget will appear.")
    notebook_login()


## Step 4 - Build the shared evaluation datasets

Every model is evaluated on the same prompts, datasets, train split, test split, probe method, and layer rule. This makes the comparison fair.

`MAX_HF_GROUPS` controls the number of Hugging Face benchmark groups. Increase it for a stronger final run, but start small to verify the runtime.


In [ ]:
from lie_detector_llm.datasets import build_dataset_collection

INCLUDE_HF_DATASETS = True
MAX_HF_GROUPS = 25

collection = build_dataset_collection(
    include_hf_datasets=INCLUDE_HF_DATASETS,
    max_hf_groups=MAX_HF_GROUPS,
)
dataset_names = collection.dataset_names()

stats = (
    collection.frame.groupby("dataset_name")
    .agg(rows=("dataset_name", "size"), groups=("group_id", "nunique"))
    .reset_index()
    .sort_values("dataset_name")
)

display(stats)
print(f"Total prompts: {len(collection.frame)}")
print(f"Total groups : {collection.frame['group_id'].nunique()}")


## Step 5 - Choose models to compare

The 70B models are enabled automatically only if the runtime has at least 70 GB of GPU memory. You can force them by setting `RUN_70B=True`, but that may cause an out-of-memory error on smaller Colab GPUs.

All 7B/8B/70B models use 4-bit loading to reduce memory usage.


In [ ]:
RUN_70B = GPU_TOTAL_GB >= 70
FORCE_RUN_70B = False
RUN_70B = RUN_70B or FORCE_RUN_70B

MODEL_CONFIGS = [
    {
        "label": "Phi-2 baseline",
        "model_name": "microsoft/phi-2",
        "load_in_4bit": False,
        "enabled": True,
    },
    {
        "label": "Mistral 7B Instruct",
        "model_name": "mistralai/Mistral-7B-Instruct-v0.3",
        "load_in_4bit": True,
        "enabled": True,
    },
    {
        "label": "Llama 3 8B Instruct",
        "model_name": "meta-llama/Meta-Llama-3-8B-Instruct",
        "load_in_4bit": True,
        "enabled": True,
    },
    {
        "label": "Llama 3.3 70B Instruct",
        "model_name": "meta-llama/Llama-3.3-70B-Instruct",
        "load_in_4bit": True,
        "enabled": RUN_70B,
    },
    {
        "label": "Llama 3 70B Instruct",
        "model_name": "meta-llama/Meta-Llama-3-70B-Instruct",
        "load_in_4bit": True,
        "enabled": RUN_70B,
    },
]

active_configs = [config for config in MODEL_CONFIGS if config["enabled"]]
print("Models that will run:")
for config in active_configs:
    print(f"- {config['label']}: {config['model_name']} (4-bit={config['load_in_4bit']})")

if not RUN_70B:
    print()
    print("70B models are currently disabled. Set FORCE_RUN_70B=True if you have enough GPU memory.")

## Step 6 - Configure the shared probing experiment

We use one fixed setup for all models:

- train dataset: `repeng_truthful`,
- evaluation: test split of every dataset,
- probe method: logistic regression,
- layer index: `-1`, the final transformer layer,
- activation batch size: 1, safer for large models.

A stronger final report can add a separate layer sweep per model, but this fixed-layer comparison is the cleanest first multi-LLM result.


In [ ]:
TRAIN_DATASET = "repeng_truthful"
PROBE_METHOD = "lr"
LAYER_INDEX = -1
ACTIVATION_BATCH_SIZE = 1
SPLIT_EVALUATION = True

print("Shared experiment settings")
print(f"Train dataset : {TRAIN_DATASET}")
print(f"Eval datasets : {dataset_names}")
print(f"Probe method  : {PROBE_METHOD}")
print(f"Layer index   : {LAYER_INDEX}")


## Step 7 - Run the comparison loop

Each model is loaded, evaluated, and then removed from the model cache before the next model starts. This is important on Colab because GPU memory is limited.


In [ ]:
import gc
import time
import pandas as pd
import torch

from lie_detector_llm.experiment import run_transfer_experiment
from lie_detector_llm.models import clear_model_cache

all_results = []
failures = []

for config in active_configs:
    print("=" * 80)
    print(f"Running {config['label']} ({config['model_name']})")
    start = time.perf_counter()
    try:
        output = run_transfer_experiment(
            collection=collection,
            train_dataset_name=TRAIN_DATASET,
            eval_dataset_names=dataset_names,
            model_name=config["model_name"],
            probe_method=PROBE_METHOD,
            layer_index=LAYER_INDEX,
            split_evaluation=SPLIT_EVALUATION,
            activation_batch_size=ACTIVATION_BATCH_SIZE,
            load_in_4bit=config["load_in_4bit"],
            show_progress=True,
        )
        df_model = output.summary_table()
        df_model["model_label"] = config["label"]
        df_model["load_in_4bit"] = config["load_in_4bit"]
        df_model["runtime_seconds"] = time.perf_counter() - start
        all_results.append(df_model)
        display(df_model.sort_values("eval_dataset"))
    except Exception as exc:
        failures.append(
            {
                "model_label": config["label"],
                "model_name": config["model_name"],
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
        )
        print(f"FAILED: {type(exc).__name__}: {exc}")
    finally:
        clear_model_cache()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if not all_results:
    raise RuntimeError("No model completed successfully.")

comparison = pd.concat(all_results, ignore_index=True)
failures_df = pd.DataFrame(failures)


## Step 8 - Build the comparison table


In [ ]:
pivot = comparison.pivot_table(
    index="model_label",
    columns="eval_dataset",
    values="grouped_accuracy",
).round(3)

display(pivot)

transfer_only = comparison[comparison["eval_dataset"] != TRAIN_DATASET]
summary = (
    transfer_only.groupby(["model_label", "model_name"], as_index=False)
    .agg(
        mean_transfer_accuracy=("grouped_accuracy", "mean"),
        min_transfer_accuracy=("grouped_accuracy", "min"),
        max_transfer_accuracy=("grouped_accuracy", "max"),
        runtime_seconds=("runtime_seconds", "max"),
    )
    .sort_values("mean_transfer_accuracy", ascending=False)
)

display(summary)

if len(failures_df):
    print("Models that failed:")
    display(failures_df)


## Step 9 - Plot model comparison


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=transfer_only,
    x="eval_dataset",
    y="grouped_accuracy",
    hue="model_label",
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_title("Cross-dataset transfer accuracy by model")
ax.set_xlabel("Evaluation dataset")
ax.set_ylabel("Grouped accuracy")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig


## Step 10 - Save the results


In [ ]:
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

comparison_path = results_dir / "large_llm_comparison.csv"
summary_path = results_dir / "large_llm_comparison_summary.csv"
failures_path = results_dir / "large_llm_comparison_failures.csv"

comparison.to_csv(comparison_path, index=False)
summary.to_csv(summary_path, index=False)
failures_df.to_csv(failures_path, index=False)

print(f"Saved detailed comparison to: {comparison_path}")
print(f"Saved summary to            : {summary_path}")
print(f"Saved failures to           : {failures_path}")


## Step 11 - Optional full matrix for the best model

The comparison above trains only on `repeng_truthful`. After identifying the best model, you can run a full train-dataset by eval-dataset matrix for that model.

This is slower, so it is disabled by default.


In [ ]:
RUN_FULL_MATRIX_FOR_BEST = False

if RUN_FULL_MATRIX_FOR_BEST:
    from lie_detector_llm.experiment import run_full_transfer_matrix
    from lie_detector_llm.plotting import plot_transfer_heatmap

    best_row = summary.iloc[0]
    best_config = next(
        config for config in active_configs if config["label"] == best_row["model_label"]
    )

    matrix = run_full_transfer_matrix(
        collection=collection,
        model_name=best_config["model_name"],
        probe_method=PROBE_METHOD,
        layer_index=LAYER_INDEX,
        split_evaluation=True,
        activation_batch_size=ACTIVATION_BATCH_SIZE,
        load_in_4bit=best_config["load_in_4bit"],
        show_progress=True,
    )

    display(matrix.summary_table())
    fig, ax = plot_transfer_heatmap(
        matrix.results,
        title=f"Full transfer matrix: {best_config['label']}",
    )
    display(fig)

    clear_model_cache()


## How to report these results

A good course-report paragraph should state:

- which models were compared,
- which datasets were used,
- which probe and layer were fixed for fairness,
- the average off-domain grouped accuracy for each model,
- whether larger instruction-tuned models improved transfer.

The key table is `large_llm_comparison_summary.csv`. The key plot is the bar chart from Step 9.
